In [ ]:
from data_pipeline import load_and_prepare, engineer_features, stratified_group_split

PRQT_FILE_PATH = r"\\bosch.com\dfsrb\DfsDE\LOC\Rt\BST\09_projects\BMI420\External\02_Product_Development\03_System\04_Accel_System_Development\09_FT_Evaluations\BAI_CA\Possible_slope_correlation_CP3_off\acc_std_mean_CP2_CP3_CP4.parquet"

targets = ['hot_part_fail', 'cold_part_fail']

pivoted = load_and_prepare(PRQT_FILE_PATH)
data, features_clean = engineer_features(pivoted, targets)
split = stratified_group_split(data, targets)

# PREDICTION

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, accuracy_score, confusion_matrix,
                              f1_score, precision_recall_curve, auc)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
import json
import os
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Business cost parameters ---
FN_COST = 10
FP_COST = 1
RECALL_AT_PRECISION_TARGET = 0.8
N_OPTUNA_TRIALS = 50
OPTUNA_PARAMS_FILE = 'optuna_best_params.json'

# --- Unpack split ---
train_idx = split['train_idx']
val_idx = split['val_idx']
holdout_idx = split['holdout_idx']

X = data[features_clean].values

# --- Load or run Optuna HPO ---
def make_objective(X_tr, y_tr, X_v, y_v, sw, fail_idx):
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 2, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss',
                          early_stopping_rounds=20)
        m.fit(X_tr, y_tr, sample_weight=sw, eval_set=[(X_v, y_v)], verbose=False)
        proba = m.predict_proba(X_v)[:, fail_idx]
        best_cost = float('inf')
        for th in np.arange(0.05, 0.96, 0.05):
            pred = (proba >= th).astype(int)
            if fail_idx != 1:
                pred = 1 - pred
            cm_t = confusion_matrix(y_v, pred, labels=[0, 1])
            tn_, fp_, fn_, tp_ = cm_t.ravel()
            best_cost = min(best_cost, FN_COST * fn_ + FP_COST * fp_)
        return best_cost
    return objective

# Try loading saved hyperparameters
saved_params = {}
if os.path.exists(OPTUNA_PARAMS_FILE):
    with open(OPTUNA_PARAMS_FILE, 'r') as f:
        saved_params = json.load(f)
    print(f"\nLoaded saved Optuna params from {OPTUNA_PARAMS_FILE}")

results = {}
new_params = {}
for target in targets:
    print(f"\n{'='*60}")
    print(f"Target: {target}")
    print(f"{'='*60}")

    le = LabelEncoder()
    y_all = le.fit_transform(data[target].values)
    fail_class_idx = list(le.classes_).index('1')

    X_train, y_train = X[train_idx], y_all[train_idx]
    X_val, y_val = X[val_idx], y_all[val_idx]
    X_holdout, y_holdout = X[holdout_idx], y_all[holdout_idx]
    sample_weights = compute_sample_weight('balanced', y_train)

    if target in saved_params:
        bp = saved_params[target]
        print(f"  Using saved params: {bp}")
    else:
        # Optuna search
        study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(make_objective(X_train, y_train, X_val, y_val,
                                      sample_weights, fail_class_idx),
                       n_trials=N_OPTUNA_TRIALS)
        bp = study.best_params
        print(f"  Optuna best val cost: {study.best_value}")
        print(f"  Best params: {bp}")

    new_params[target] = bp

    # Train final model
    model = XGBClassifier(**bp, random_state=42, eval_metric='logloss',
                          early_stopping_rounds=20)
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    print(f"  Early stopping: iteration {model.best_iteration} / {bp['n_estimators']}")

    # Threshold sweep on validation set
    y_val_proba = model.predict_proba(X_val)[:, fail_class_idx]
    sweep_results = []
    for th in np.arange(0.05, 0.96, 0.01):
        y_pred_th = (y_val_proba >= th).astype(int)
        if fail_class_idx != 1:
            y_pred_th = 1 - y_pred_th
        cm_th = confusion_matrix(y_val, y_pred_th, labels=[0, 1])
        tn, fp, fn, tp = cm_th.ravel()
        sweep_results.append({
            'threshold': th, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'recall': tp / (tp + fn) if (tp + fn) > 0 else 0,
            'precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
            'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
            'cost': FN_COST * fn + FP_COST * fp,
        })
    best = min(sweep_results, key=lambda x: x['cost'])
    THRESHOLD = round(best['threshold'], 2)
    print(f"  Val threshold: {THRESHOLD} (cost={best['cost']}, FN={best['FN']}, FP={best['FP']})")

    # Holdout evaluation
    y_hold_proba = model.predict_proba(X_holdout)[:, fail_class_idx]
    y_hold_pred = (y_hold_proba >= THRESHOLD).astype(int)
    if fail_class_idx != 1:
        y_hold_pred = 1 - y_hold_pred

    hold_labels = sorted(set(y_holdout) | set(y_hold_pred))
    hold_names = [str(le.classes_[i]) for i in hold_labels]
    acc = accuracy_score(y_holdout, y_hold_pred)
    f1_macro = f1_score(y_holdout, y_hold_pred, average='macro', labels=hold_labels)
    f1_weighted = f1_score(y_holdout, y_hold_pred, average='weighted', labels=hold_labels)
    cm = confusion_matrix(y_holdout, y_hold_pred, labels=hold_labels)
    report = classification_report(y_holdout, y_hold_pred, labels=hold_labels,
                                   target_names=hold_names, output_dict=True)
    cm_hold = confusion_matrix(y_holdout, y_hold_pred, labels=[0, 1])
    hold_tn, hold_fp, hold_fn, hold_tp = cm_hold.ravel()
    holdout_cost = int(FN_COST * hold_fn + FP_COST * hold_fp)

    hold_pr_prec, hold_pr_rec, hold_pr_th = precision_recall_curve(
        y_holdout, y_hold_proba, pos_label=fail_class_idx)
    pr_auc = auc(hold_pr_rec, hold_pr_prec)
    mask_prec = hold_pr_prec >= RECALL_AT_PRECISION_TARGET
    recall_at_prec = float(hold_pr_rec[mask_prec].max()) if mask_prec.any() else 0.0

    val_pr_prec, val_pr_rec, val_pr_th = precision_recall_curve(
        y_val, y_val_proba, pos_label=fail_class_idx)

    print(f"\n  Holdout: Acc={acc:.4f} F1M={f1_macro:.4f} PR-AUC={pr_auc:.4f} "
          f"Recall@{RECALL_AT_PRECISION_TARGET}prec={recall_at_prec:.4f}")
    print(f"  Cost={holdout_cost} (FN={hold_fn}×{FN_COST} + FP={hold_fp}×{FP_COST})")

    results[target] = {
        'model': model, 'le': le, 'accuracy': acc, 'f1_macro': f1_macro,
        'f1_weighted': f1_weighted, 'confusion_matrix': cm, 'class_names': hold_names,
        'report': report, 'feature_importances': model.feature_importances_,
        'pr_precision': val_pr_prec, 'pr_recall': val_pr_rec, 'pr_thresholds': val_pr_th,
        'hold_pr_precision': hold_pr_prec, 'hold_pr_recall': hold_pr_rec,
        'hold_pr_thresholds': hold_pr_th,
        'pr_auc': pr_auc, 'recall_at_prec': recall_at_prec,
        'recall_at_prec_target': RECALL_AT_PRECISION_TARGET,
        'threshold': THRESHOLD, 'sweep_results': sweep_results,
        'fn_cost': FN_COST, 'fp_cost': FP_COST,
        'holdout_cost': holdout_cost, 'holdout_fn': int(hold_fn), 'holdout_fp': int(hold_fp),
        'val_cost': int(best['cost']), 'features_clean': features_clean,
        'best_iteration': model.best_iteration, 'best_params': bp,
    }

# Save params (write after all targets succeed)
with open(OPTUNA_PARAMS_FILE, 'w') as f:
    json.dump(new_params, f, indent=2)
print(f"\nSaved Optuna best params to {OPTUNA_PARAMS_FILE}")

total_cost = sum(results[t]['holdout_cost'] for t in targets)
print(f"Total holdout cost: {total_cost}")
print(f"Models trained with Optuna-optimized hyperparameters + sample_weight balancing.")

Feature reduction: 50 -> 19 (0 near-constant, 31 correlated)

Per-wafer fail rates:
  DPK456-11-C0 (7366 dies): hot_part_fail: 0.042 | cold_part_fail: 0.044
  DPK456-12-E3 (7372 dies): hot_part_fail: 0.032 | cold_part_fail: 0.036
  DPK456-13-G6 (7361 dies): hot_part_fail: 0.050 | cold_part_fail: 0.054
  DPK456-14-B6 (7356 dies): hot_part_fail: 0.033 | cold_part_fail: 0.038
  DPK456-15-E1 (7361 dies): hot_part_fail: 0.050 | cold_part_fail: 0.053
  DPK456-16-G4 (7384 dies): hot_part_fail: 0.034 | cold_part_fail: 0.037
  DPK456-17-B4 (7367 dies): hot_part_fail: 0.039 | cold_part_fail: 0.042

Stratified split:
  Train:   22101 dies / 3 wafers ['DPK456-13-G6', 'DPK456-14-B6', 'DPK456-16-G4']
  Val:     14733 dies / 2 wafers ['DPK456-11-C0', 'DPK456-17-B4']
  Holdout: 14733 dies / 2 wafers ['DPK456-12-E3', 'DPK456-15-E1']
  hot_part_fail fail rate — Train: 0.0390 | Val: 0.0400 | Holdout: 0.0411
  cold_part_fail fail rate — Train: 0.0426 | Val: 0.0432 | Holdout: 0.0445

Loaded saved Optuna pa

In [ ]:
from model_dashboard import run_dashboard

run_dashboard(
    results, targets, features_clean,
    title='XGBoost Model Performance Report',
    subtitle='Predicting cold/hot part_fail from CP2 test features | Threshold selected by business cost on validation set',
    port=8051,
)